In [ ]:
BUILD_SRC = "\"\"\"Build a harness-ready task pool from the G8 validation results (runs in an INTERNET-enabled CPU Kaggle notebook; dry-run locally with --limit).\n\nOutput layout (the competition's, so swegemma's Evaluator / verify_task can consume it unchanged):\n  <out>/tasks.jsonl                    instance_id, repo, base_commit, problem_statement, hints_text, patch, test_patch, created_at, FAIL_TO_PASS, PASS_TO_PASS, environment_setup_commit\n  <out>/snapshots/<instance_id>.tgz    repo at base_commit rooted at '.', WITH a .git directory (HEAD -> refs/heads/main), like the competition's snapshots\n  <out>/wheels/*.whl                   shared wheel set: highest version per package over the usable tasks' freeze lists, minus packages the Kaggle image already has at >= that version\n  <out>/manifest.json                  per-task dependency status, dropped packages, sizes\n\nUsage: python build_pool.py <results_dir_or_files...> --candidates <candidates.jsonl> --out <dir> [--limit N]\n\"\"\"\n\nimport argparse\nimport concurrent.futures as cf\nimport importlib.metadata as md\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tarfile\nimport time\nfrom pathlib import Path\n\nfrom packaging.version import InvalidVersion, Version\n\nMAX_SNAPSHOT_MB = 150\nMAX_WHEEL_MB = 150\n\n\ndef run(cmd, cwd=None, timeout=600, env=None):\n    try:\n        p = subprocess.run(cmd, cwd=cwd, env=env, capture_output=True, text=True, timeout=timeout, errors=\"replace\")\n        return p.returncode, (p.stdout or \"\") + (p.stderr or \"\")\n    except subprocess.TimeoutExpired:\n        return -999, \"timeout\"\n\n\ndef norm(name):\n    return re.sub(r\"[-_.]+\", \"-\", name).lower()\n\n\ndef make_snapshot(task, out_dir, work):\n    iid, repo = task[\"instance_id\"], task[\"repo\"]\n    d = Path(work) / iid\n    shutil.rmtree(d, ignore_errors=True)\n    d.mkdir(parents=True)\n    env = dict(os.environ, GIT_TERMINAL_PROMPT=\"0\")\n    for cmd in ([\"git\", \"init\", \"-q\", \"-b\", \"main\"], [\"git\", \"remote\", \"add\", \"origin\", f\"https://github.com/{repo}.git\"]):\n        run(cmd, d, 120, env)\n    rc, out = run([\"git\", \"fetch\", \"-q\", \"--depth\", \"1\", \"origin\", task[\"base_commit\"]], d, 900, env)\n    if rc != 0:\n        rc, out = run([\"git\", \"fetch\", \"-q\", \"origin\", task[\"base_commit\"]], d, 1500, env)\n    if rc == 0:\n        rc, out = run([\"git\", \"checkout\", \"-q\", \"-B\", \"main\", \"FETCH_HEAD\"], d, 300, env)\n    if rc != 0:\n        shutil.rmtree(d, ignore_errors=True)\n        return {\"ok\": False, \"reason\": \"git: \" + out[-200:]}\n    tgz = Path(out_dir) / \"snapshots\" / f\"{iid}.tgz\"\n    with tarfile.open(tgz, \"w:gz\", compresslevel=3) as t:\n        t.add(d, arcname=\".\")\n    mb = tgz.stat().st_size / 1e6\n    shutil.rmtree(d, ignore_errors=True)\n    if mb > MAX_SNAPSHOT_MB:\n        tgz.unlink()\n        return {\"ok\": False, \"reason\": f\"snapshot {mb:.0f} MB > {MAX_SNAPSHOT_MB}\"}\n    return {\"ok\": True, \"mb\": round(mb, 1)}\n\n\ndef host_versions():\n    out = {}\n    for dist in md.distributions():\n        n = dist.metadata[\"Name\"]\n        if n:\n            out[norm(n)] = dist.version\n    return out\n\n\ndef parse_freeze(lines):\n    deps = {}\n    for l in lines:\n        l = l.strip()\n        if not l or l.startswith(\"-e\") or \" @ \" in l or \"==\" not in l:\n            continue\n        n, v = l.split(\"==\", 1)\n        deps[norm(n)] = v.split(\";\")[0].strip()\n    return deps\n\n\ndef vkey(v):\n    try:\n        return Version(v)\n    except InvalidVersion:\n        return Version(\"0\")\n\n\ndef fetch_wheel(name, versions, wheels_dir):\n    \"\"\"Try the highest version first, then lower ones seen in freezes; wheel first, build from sdist as a fallback.\"\"\"\n    for v in versions[:3]:\n        rc, out = run([sys.executable, \"-m\", \"pip\", \"download\", \"-q\", \"--no-deps\", \"--only-binary=:all:\", \"-d\", str(wheels_dir), f\"{name}=={v}\"], timeout=300)\n        if rc == 0:\n            return {\"ok\": True, \"version\": v, \"how\": \"wheel\"}\n    for v in versions[:2]:\n        rc, out = run([sys.executable, \"-m\", \"pip\", \"wheel\", \"-q\", \"--no-deps\", \"-w\", str(wheels_dir), f\"{name}=={v}\"], timeout=420)\n        if rc == 0:\n            return {\"ok\": True, \"version\": v, \"how\": \"built\"}\n    return {\"ok\": False, \"reason\": out[-160:]}\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"results\", nargs=\"+\")\n    ap.add_argument(\"--candidates\", required=True)\n    ap.add_argument(\"--out\", required=True)\n    ap.add_argument(\"--limit\", type=int, default=0)\n    ap.add_argument(\"--workers\", type=int, default=6)\n    a = ap.parse_args()\n    out = Path(a.out)\n    (out / \"snapshots\").mkdir(parents=True, exist_ok=True)\n    (out / \"wheels\").mkdir(exist_ok=True)\n    rows = []\n    for p in a.results:\n        rows += json.load(open(p, encoding=\"utf-8\"))\n    cand = {}\n    for l in open(a.candidates, encoding=\"utf-8\"):\n        t = json.loads(l)\n        cand[t[\"instance_id\"]] = t\n    usable = [r for r in rows if r[\"verdict\"] in (\"valid\", \"p2p_partial\") and r[\"id\"] in cand]\n    if a.limit:\n        usable = usable[: a.limit]\n    print(f\"{len(rows)} results, {len(usable)} usable\", flush=True)\n    t0 = time.time()\n    manifest = {\"tasks\": {}, \"wheels\": {}, \"oversize_wheels\": [], \"usable\": len(usable)}\n\n    # ---- 1. snapshots\n    work = Path(os.environ.get(\"TMPDIR\", \"/tmp\")) / \"g9_snap\"\n    work.mkdir(parents=True, exist_ok=True)\n    snap = {}\n    with cf.ThreadPoolExecutor(a.workers) as ex:\n        futs = {ex.submit(make_snapshot, cand[r[\"id\"]], out, work): r[\"id\"] for r in usable}\n        for k, f in enumerate(cf.as_completed(futs), 1):\n            snap[futs[f]] = f.result()\n            if k % 25 == 0 or k == len(futs):\n                print(f\"[{time.time() - t0:5.0f}s] snapshots {k}/{len(futs)} ok={sum(1 for v in snap.values() if v['ok'])}\", flush=True)\n    kept = [r for r in usable if snap[r[\"id\"]][\"ok\"]]\n\n    # ---- 2. shared wheel set\n    host = host_versions()\n    want = {}   # name -> sorted list of versions seen (high to low)\n    task_deps = {}\n    for r in kept:\n        deps = parse_freeze(r.get(\"freeze\", []))\n        task_deps[r[\"id\"]] = deps\n        for n, v in deps.items():\n            want.setdefault(n, set()).add(v)\n    need = {}\n    for n, vs in want.items():\n        vs = sorted(vs, key=vkey, reverse=True)\n        hv = host.get(n)\n        if hv is not None and vkey(hv) >= vkey(vs[0]):\n            continue   # the Kaggle image already has this package at least as new\n        need[n] = vs\n    print(f\"{len(want)} distinct packages in freezes; {len(need)} not covered by the host image -> downloading\", flush=True)\n    res = {}\n    with cf.ThreadPoolExecutor(8) as ex:\n        futs = {ex.submit(fetch_wheel, n, vs, out / \"wheels\"): n for n, vs in need.items()}\n        for k, f in enumerate(cf.as_completed(futs), 1):\n            res[futs[f]] = f.result()\n            if k % 50 == 0 or k == len(futs):\n                print(f\"[{time.time() - t0:5.0f}s] wheels {k}/{len(futs)} ok={sum(1 for v in res.values() if v['ok'])}\", flush=True)\n    for w in list((out / \"wheels\").glob(\"*.whl\")):\n        if w.stat().st_size > MAX_WHEEL_MB * 1e6:\n            manifest[\"oversize_wheels\"].append(w.name)\n            w.unlink()\n    manifest[\"wheels\"] = {n: v for n, v in res.items()}\n    bad_pkgs = {n for n, v in res.items() if not v[\"ok\"]} | {norm(re.split(r\"[-_]\\d\", w)[0]) for w in manifest[\"oversize_wheels\"]}\n\n    # ---- 3. tasks.jsonl + manifest\n    n_complete = 0\n    with open(out / \"tasks.jsonl\", \"w\", encoding=\"utf-8\") as fh:\n        for r in kept:\n            t = cand[r[\"id\"]]\n            missing = sorted(n for n in task_deps[r[\"id\"]] if n in bad_pkgs)\n            manifest[\"tasks\"][r[\"id\"]] = {\"verdict\": r[\"verdict\"], \"snapshot_mb\": snap[r[\"id\"]][\"mb\"], \"missing_deps\": missing, \"n_deps\": len(task_deps[r[\"id\"]])}\n            n_complete += not missing\n            fh.write(json.dumps({k: t[k] for k in (\"instance_id\", \"repo\", \"base_commit\", \"problem_statement\", \"hints_text\", \"patch\", \"test_patch\", \"created_at\",\n                                                   \"FAIL_TO_PASS\", \"PASS_TO_PASS\", \"environment_setup_commit\")}) + \"\\n\")\n    manifest[\"snapshots_ok\"] = len(kept)\n    manifest[\"tasks_with_complete_deps\"] = n_complete\n    (out / \"manifest.json\").write_text(json.dumps(manifest, indent=1), encoding=\"utf-8\")\n    wsize = sum(w.stat().st_size for w in (out / \"wheels\").glob(\"*.whl\")) / 1e9\n    ssize = sum(s.stat().st_size for s in (out / \"snapshots\").glob(\"*.tgz\")) / 1e9\n    print(f\"\\nDONE in {time.time() - t0:.0f}s | tasks {len(kept)} (complete deps {n_complete}) | snapshots {ssize:.2f} GB | wheels {len(list((out / 'wheels').glob('*.whl')))} files {wsize:.2f} GB \"\n          f\"| failed packages {len(bad_pkgs)} | oversize wheels {len(manifest['oversize_wheels'])}\", flush=True)\n    if bad_pkgs:\n        print(\"packages without a usable wheel (first 30):\", sorted(bad_pkgs)[:30])\n\n\nif __name__ == \"__main__\":\n    main()\n"

In [ ]:
import subprocess, sys, os, glob
from pathlib import Path
Path('/kaggle/working/build_pool.py').write_text(BUILD_SRC, encoding='utf-8')
res = sorted(glob.glob('/kaggle/input/**/scale_results_*.json', recursive=True))
cand = glob.glob('/kaggle/input/**/candidates.jsonl', recursive=True)
print('result files:', res, '| candidates:', cand)
assert len(res) == 2 and len(cand) == 1, (res, cand)
rc = subprocess.run([sys.executable, '-u', '/kaggle/working/build_pool.py', *res, '--candidates', cand[0], '--out', '/kaggle/working/pool', '--workers', '6']).returncode
print('build_pool exit code', rc)
assert rc == 0